# Crypto Portfolio Optimization V4 — Walk-forward

Cách dùng: **chạy cell thư viện → chỉnh 2 cell PARAMS → chạy tuần tự các cell còn lại.** Mọi tham số backtest nằm trong 2 cell PARAMS.

Khác V3 (V3 giữ nguyên, không sửa):
- **Walk-forward**: tại mỗi candle `t`, fit trên `n_window` candle liền trước (không gồm `t`) → weight áp cho return của candle `t`. Tải `display_candles + n_window` nến: nửa đầu chỉ làm cửa sổ fit, nửa sau là phần backtest hiển thị.
- **Hard-rule trễ dữ liệu 1 ngày** (`data_lag_days`): hôm nay là T chỉ có data đến T-1, weight tối ưu từ đó chỉ ảnh hưởng từ T+1.
- Không `bfill` (chặn look-ahead); coin chưa đủ dữ liệu trong cửa sổ bị loại khỏi cửa sổ đó.
- Chọn tài sản / tối ưu / xếp hạng chỉ dùng dữ liệu cửa sổ. Grid `max_assets` vẫn có nhưng mỗi giá trị được walk-forward riêng.
- `mu` arithmetic (không ghép lãi theo giờ) và co về trung bình chéo (`mu_shrink`).
- Một `RF` cho optimizer + metrics; có phí giao dịch trên turnover.

In [ ]:
!pip install --upgrade binance-historical-data PyPortfolioOpt -q

In [ ]:
import time
start_time = time.time()

import os
import glob
import requests
import zipfile
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from math import ceil
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from rich.progress import Progress
from rich.console import Console
from typing import List
from pypfopt import EfficientFrontier, CovarianceShrinkage, risk_models

# Tải dữ liệu thô từ Binance + lấy top coin (giữ nguyên từ V3)
def date_range_daily(start: str, end: str):
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    current = start_dt
    dates = []
    while current <= end_dt:
        dates.append(current.strftime("%Y-%m-%d"))
        current += relativedelta(days=1)
    return dates

def date_range_monthly(start: str, end: str):
    start_dt = datetime.strptime(start, "%Y-%m")
    end_dt = datetime.strptime(end, "%Y-%m")
    current = start_dt
    months = []
    while current <= end_dt:
        months.append(current.strftime("%Y-%m"))
        current += relativedelta(months=1)
    return months

def download_binance_klines_fixed_symbols(
    symbols: List[str],
    intervals: List[str],
    time_period: str,          # "daily" or "monthly"
    start: str,                # "2023-01" or "2023-01-01"
    end: str,                  # "2025-11" or "2025-12-10"
    destination_dir: str = "./binance_data",
    max_workers: int = 15,
    retries: int = 3
):
    console = Console()
    base_url = "https://data.binance.vision/data"
    asset_path = "spot"

    if time_period == "daily":
        date_list = date_range_daily(start, end)
        url_template = f"{base_url}/{asset_path}/{time_period}/klines/{{symbol}}/{{interval}}/{{symbol}}-{{interval}}-{{date}}.zip"
        csv_name_template = "{symbol}-{interval}-{date}.csv"
    elif time_period == "monthly":
        date_list = date_range_monthly(start, end)
        url_template = f"{base_url}/{asset_path}/{time_period}/klines/{{symbol}}/{{interval}}/{{symbol}}-{{interval}}-{{date}}.zip"
        csv_name_template = "{symbol}-{interval}-{date}.csv"
    else:
        raise ValueError("time_period must be 'daily' or 'monthly'")

    # Chỉ tải file còn thiếu: đã có CSV (hoặc marker .404 = Binance không có file) thì bỏ qua
    stale_before = ((date.today() - timedelta(days=5)).strftime("%Y-%m-%d") if time_period == "daily"
                    else date.today().strftime("%Y-%m"))
    def _csv_path(s, i, d):
        return os.path.join(destination_dir, asset_path, s, i, csv_name_template.format(symbol=s, interval=i, date=d))
    todo = [(s, i, d) for s in symbols for i in intervals for d in date_list
            if not os.path.exists(_csv_path(s, i, d)) and not os.path.exists(_csv_path(s, i, d) + ".404")]
    total_files = len(todo)
    if not todo:
        console.print("[bold green]Đã đủ data, bỏ qua bước tải.[/]")
        return
    success_count = 0

    def download_one(symbol: str, interval: str, date_str: str):
        nonlocal success_count
        url = url_template.format(symbol=symbol, interval=interval, date=date_str)
        csv_name = csv_name_template.format(symbol=symbol, interval=interval, date=date_str)
        local_dir = os.path.join(destination_dir, asset_path, symbol, interval)
        local_csv = os.path.join(local_dir, csv_name)
        local_zip = local_csv.replace(".csv", ".zip")

        os.makedirs(local_dir, exist_ok=True)

        # Skip if already have CSV
        if os.path.exists(local_csv):
            return True

        for attempt in range(retries + 1):
            try:
                resp = requests.get(url, timeout=20)
                if resp.status_code == 404:
                    if date_str < stale_before:       # ngày đã cũ mà vẫn 404 = không tồn tại -> đánh dấu, lần sau khỏi hỏi lại
                        open(local_csv + ".404", "w").close()
                    return False
                resp.raise_for_status()

                # Save ZIP
                with open(local_zip, "wb") as f:
                    f.write(resp.content)

                # Extract first CSV
                with zipfile.ZipFile(local_zip) as zf:
                    for name in zf.namelist():
                        if name.endswith(".csv"):
                            zf.extract(name, local_dir)
                            os.rename(os.path.join(local_dir, name), local_csv)
                            break

                os.remove(local_zip)  # optional: save disk space
                return True

            except Exception as e:
                if attempt == retries:
                    console.print(f"[red]Failed {url}: {e}[/]")
                    return False
        return False

    console.print(f"[bold blue]Total files to attempt: {total_files}[/]")

    with Progress() as progress:
        task = progress.add_task("[cyan]Downloading...", total=total_files)
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = []
            for symbol, interval, d in todo:
                futures.append(executor.submit(download_one, symbol, interval, d))

            for future in as_completed(futures):
                if future.result():
                    success_count += 1
                progress.advance(task)

    console.print(f"[bold green]Done. Successfully downloaded {success_count}/{total_files} files.[/]")

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def read_csv_file(file_path):
    # File Binance KHÔNG có dòng header (V3 dùng header mặc định nên mất dòng đầu mỗi file = nến 00:00).
    df = pd.read_csv(file_path, header=None)
    df = df[pd.to_numeric(df[0], errors="coerce").notna()].astype({0: "int64", 6: "int64"})   # bỏ header nếu có
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    if df.empty:
        return df
    unit = 'ms' if len(str(df['open_time'].iloc[0])) == 13 else 'us'   # vectorize: nhanh hơn apply từng dòng
    df['open_time'] = pd.to_datetime(df['open_time'], unit=unit)
    df['close_time'] = pd.to_datetime(df['close_time'], unit=unit)
    return df

def get_top_coins(n_coins: int = 10) -> list:
    """
    Lấy danh sách top N đồng coin theo vốn hóa thị trường từ CoinGecko,
    loại bỏ các stablecoin, token không mong muốn và symbol trùng lặp.

    Parameters:
    - n_coins (int): Số lượng coin muốn lấy (mặc định: 1 0)

    Returns:
    - List[str]: Danh sách symbol (viết hoa), không trùng lặp, không trong blacklist
    """
    blacklist = {
        "USDT", "USDC", "DAI", "TUSD", "UST", "WBTC", "STETH", "FIGR_HELOC", "WBT", "WSTETH",
        "WBETH", "USDS", "LEO", "BSC-USD", "WETH", "FDUSD", "USDP", "GUSD",
        "WEETH", "XMR", "USDE", "CBBTC", "SUSDS", "CRO", "USDT0", "PYUSD", "SUSDE", "HASH",
        "MNT", "USD1", "CC", "BGB", "OKB", "M", "USDF", "BUIDL", "JITOSOL",
        "PI", "RAIN", "JLP", "HTX", "SYRUPUSDC", "USYC", "USDG", "JSOL",
        "KCS", "RLUSD", "RSETH", "BNSOL", "WBNB", "BFUSD", "RIVER", "GT", "USDD", "RETH",
        "MYX", "KAS", "LBTC", "FBTC", "USDTB", "USTB", "USDC.E", "FLR", "SYRUPUSDT", "METH",
        "WM", "JAAA", "LSETH", "KHYPE", "USDY", "OUSG", "XDC", "SOLVBTC", "EUTBL", "USDAI",
        "COCA", "CLBTC", "BDX", "WFLR", "USD0", "JTRSY", "OSETH", "A7A5", "GHO", "EZETH",
        "IP", "JUPSOL", "STABLE", "FTN", "TBTC", "CTM", "EURC", "USR", "PIPPIN", "KAU",
        "YLDS", "SIREN"
    }

    per_page = min(250, max(100, n_coins * 3))  # gọi dư hơn để bù blacklist + duplicates
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": per_page,
        "page": 1,
        "sparkline": "false"
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"❌ Lỗi khi gọi API CoinGecko: {e}")
        return []

    symbols = []
    seen_symbols = set()  # theo dõi các symbol đã thêm

    for coin in data:
        symbol = coin["symbol"].upper()

        # Bỏ qua nếu đã thấy hoặc trong blacklist
        if symbol in seen_symbols or symbol in blacklist:
            continue

        symbols.append(symbol)
        seen_symbols.add(symbol)

        if len(symbols) >= n_coins:
            break

    return symbols

def get_start_date(
    timeframe: str,
    required_samples: int = 10000,
    end_date: datetime | None = None,
) -> tuple[datetime, int]:
    """
    Calculate start date to obtain at least `required_samples` candles.

    Parameters
    ----------
    timeframe : str
        e.g. '1m', '5m', '15m', '1h', '4h', '1d', '1w'
    required_samples : int
        Minimum number of candles required.
    end_date : datetime
        End date. Defaults to current UTC time.

    Returns
    -------
    start_date : datetime
    required_days : int
    """
    if end_date is None:
        end_date = datetime.utcnow()

    unit = timeframe[-1].lower()
    value = int(timeframe[:-1])

    if unit == "m":
        candles_per_day = 24 * 60 / value

    elif unit == "h":
        candles_per_day = 24 / value

    elif unit == "d":
        candles_per_day = 1 / value

    elif unit == "w":
        candles_per_day = 1 / (7 * value)

    else:
        raise ValueError(f"Unsupported timeframe: {timeframe}")

    required_days = ceil(required_samples / candles_per_day)
    start_date = end_date - timedelta(days=required_days)

    return start_date, required_days

import warnings
from joblib import Parallel, delayed, effective_n_jobs

# ====== Thư viện V4: walk-forward, không look-ahead (RF lấy từ block params) ======
def tf_delta(tf: str) -> pd.Timedelta:
    return pd.Timedelta(tf.replace('m', 'min'))


def periods_per_year(tf: str) -> float:
    return 365 * 24 * 3600 / tf_delta(tf).total_seconds()


def load_prices(tickers, start, end, timeframe, data_dir="binance_data", col="Close"):
    """Close price của mọi ticker trên 1 index chung. Chỉ ffill (causal), KHÔNG bfill:
    coin niêm yết muộn giữ NaN ở đoạn trước khi có giá."""
    delta = tf_delta(timeframe)
    idx = pd.date_range(pd.Timestamp(start, tz="UTC"),
                        pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1) - delta, freq=delta)
    def load_one(t):
        files = glob.glob(os.path.join(data_dir, "spot", t, timeframe, "*.csv"))  # mỗi file đọc 1 lần
        if not files:
            print(f"⚠️  Bỏ {t}: không có file")
            return None
        dfs = [d for d in (read_csv_file(f)[["open_time", col]].dropna() for f in files) if not d.empty]
        if not dfs:                               # bỏ file rỗng/all-NA trước khi concat (tránh FutureWarning)
            print(f"⚠️  Bỏ {t}: file rỗng")
            return None
        df = pd.concat(dfs)
        df["open_time"] = pd.to_datetime(df["open_time"], utc=True)
        s = df.drop_duplicates("open_time").set_index("open_time")[col]
        return s[s > 0].reindex(idx)

    # ponytail: tuần tự; thử ThreadPool không nhanh hơn (GIL), cần nữa thì dùng joblib theo mã
    cols = {t: s for t, s in zip(tickers, map(load_one, tickers)) if s is not None}
    return pd.DataFrame(cols).ffill()


def cov_matrix(win, P, model, span):
    shrink = lambda: CovarianceShrinkage(win, frequency=P).ledoit_wolf()
    if model == "shrinkage":
        return shrink()
    if model == "sample":
        return risk_models.sample_cov(win, frequency=P)
    S = risk_models.exp_cov(win, frequency=P, span=span)
    return S if model == "exp" else 0.5 * S + 0.5 * shrink()


def max_drawdown(r: pd.Series) -> float:
    cum = (1 + r).cumprod()
    return (cum / cum.cummax() - 1).min()


def optimize_window(win, rets, P, cfg, max_assets):
    """Chọn tài sản + tối ưu CHỈ trên `win` (dữ liệu quá khứ). Trả {strategy: weights}."""
    mu = rets.mean() * P                      # arithmetic, không ghép lãi theo giờ
    sd = rets.std() * np.sqrt(P)
    ranked = (mu / sd).sort_values(ascending=False).index

    req = {a: m for a, m in cfg["required_assets"].items() if a in win.columns}
    sel = list(dict.fromkeys(list(req) + list(ranked)))[:max_assets]
    lo = [req.get(a, cfg["min_weight"]) for a in sel]
    hi = [cfg["eur_max_weight"] if a in cfg["eur_assets"]
          else 1.0 if a in req else cfg["other_max_weight"] for a in sel]   # mã không bắt buộc bị chặn trần
    if sum(lo) > 1 or sum(hi) < 1 or any(l > h for l, h in zip(lo, hi)):    # không khả thi -> bỏ bước này
        return {}

    s = cfg["mu_shrink"]                      # co mu về trung bình chéo -> bớt nhiễu
    mu_s = (1 - s) * mu[sel] + s * mu[sel].mean()
    S = cov_matrix(win[sel], P, cfg["risk_model"], cfg["span"])
    bounds = list(zip(lo, hi))

    def solve(kind):
        ef = EfficientFrontier(mu_s, S, weight_bounds=bounds, solver_options={"adaptive_rho_interval": 25})
        try:
            if kind == "sharpe":
                ef.max_sharpe(risk_free_rate=RF)
            elif kind == "optimal":
                ef.efficient_return(cfg["target_return"])
            else:
                ef.min_volatility()
        except Exception:                     # vd: mọi mu < RF, target vượt khả thi
            ef = EfficientFrontier(mu_s, S, weight_bounds=bounds, solver_options={"adaptive_rho_interval": 25})
            ef.min_volatility()
        return dict(ef.clean_weights())

    return {k: solve(k) for k in ("sharpe", "volatility", "optimal")}


def round_weights(w, step):
    """Làm tròn weight về bội số của `step` (largest remainder): tổng giữ nguyên, không có số lẻ hơn `step`.
    Sàn/trần là bội của step nên vẫn được giữ."""
    if not step or not w:
        return w
    names = list(w)
    x = np.array([w[n] for n in names]) / step
    units = int(round(x.sum()))
    base = np.floor(x + 1e-9).astype(int)
    for i in np.argsort(-(x - base))[:max(0, units - base.sum())]:
        base[i] += 1
    return {n: float(b * step) for n, b in zip(names, base)}


def make_targets(win, P, cfg, max_assets_list, mdd_hist=None):
    """Target weight tại 1 mốc rebalance. Benchmark không phụ thuộc max_assets -> tính 1 lần;
    opt_*_m{m} tính riêng cho từng m trong max_assets_list."""
    rets = win.pct_change().dropna()
    cols = list(rets.columns)
    out = {"equal_weight": {c: 1 / len(cols) for c in cols}}

    inv = 1 / rets.std().replace(0, np.nan).dropna()
    out["risk_parity"] = (inv / inv.sum()).to_dict()

    # equal absolute drawdown: phần dư (cash_asset) không tham gia ngân sách rủi ro
    cash = cfg["cash_asset"]
    mdds = {c: abs(max_drawdown(rets[c])) for c in cols if c != cash}
    if mdd_hist:                              # MDD lịch sử dài (vd 5 năm) thay cho MDD của cửa sổ fit
        mdds = {c: mdd_hist.get(c, m) for c, m in mdds.items()}
    mdds = {c: m for c, m in mdds.items() if m > 0}
    for tgt in map(abs, cfg["mdd_targets"]):
        if not mdds:
            break
        w = {c: (tgt / len(mdds)) / m for c, m in mdds.items()}
        tot = sum(w.values())
        if tot > 1:
            w = {c: x / tot for c, x in w.items()}
            tot = 1.0
        if cash in cols:
            w[cash] = 1 - tot                 # nếu thiếu cash_asset: phần dư để tiền mặt (0%)
        out[f"absolute_equal_dd_{int(tgt * 100)}"] = w

    for m in max_assets_list:
        try:
            out.update({f"opt_{k}_m{m}": v for k, v in optimize_window(win, rets, P, cfg, m).items()})
        except Exception as e:
            print(f"  ⚠️ optimize lỗi (m={m}), giữ weight cũ: {e}")
    return out


def targets_chunk(prices, ts, n_window, timeframe, cfg, max_assets_list, lag, dd_hl):
    """Pha 1 (song song): target của mọi strategy tại các mốc rebalance `ts`. Chỉ phụ thuộc data cửa sổ, không phụ thuộc đường đi."""
    # cvxpy hay báo "Solution may be inaccurate" ở các bước tối ưu biên; kết quả vẫn dùng được -> tắt cho gọn
    warnings.filterwarnings("ignore", message="Solution may be inaccurate", category=UserWarning)
    P = periods_per_year(timeframe)
    out = []
    for t in ts:
        mdd_hist = None
        if dd_hl is not None:                     # MDD từng mã trong N năm gần nhất, chỉ dùng data đã biết
            end = prices.index[t - lag - 1] + tf_delta(timeframe)
            dh, dl = dd_hl                        # High, Low khung mdd_timeframe: đỉnh = cummax(High), đáy = Low
            m = (dh.index + tf_delta(cfg["mdd_timeframe"]) <= end) & (dh.index > end - pd.DateOffset(years=cfg["mdd_years"]))
            dh, dl = dh[m], dl[m]
            mdd_hist = {c: abs((dl[c][dh[c].notna()] / dh[c].dropna().cummax() - 1).min())
                        for c in dh.columns if dh[c].notna().sum() > 1}
        win = prices.iloc[t - lag - n_window:t - lag].dropna(axis=1)
        tg = make_targets(win, P, cfg, max_assets_list, mdd_hist)
        out.append({n: round_weights(w, cfg["weight_step"]) for n, w in tg.items()})
    return out


def walk_forward(prices, n_window, timeframe, cfg, max_assets_list, rebalance_every=1, fee=0.001, threshold=0.0, lag=0, dd_hl=None, n_jobs=-1):
    """Tại candle t: fit trên prices[t-lag-n_window : t-lag] -> áp weight cho return của candle t.
    `lag` (candle) = độ trễ dữ liệu: tại thời điểm t chỉ biết giá đến t-lag (hard-rule: dữ liệu luôn trễ 1 ngày).
    Giữa 2 lần rebalance weight trôi theo giá. Phí = fee * turnover.
    Tại mỗi mốc rebalance, strategy chỉ giao dịch nếu weight lệch >= threshold (max |target - hiện tại|).
    Pha 1: tính target mọi mốc rebalance song song (benchmark + mọi max_assets dùng chung cửa sổ/MDD).
    Pha 2: vòng candle tuần tự, vector hóa mọi strategy thành ma trận (S x A)."""
    cols = list(prices.columns)
    pos = {c: i for i, c in enumerate(cols)}
    start = n_window + lag
    total = len(prices) - start
    ts = list(range(start, len(prices), rebalance_every))

    t0 = time.time()
    chunks = np.array_split(ts, max(1, min(len(ts), 4 * effective_n_jobs(n_jobs))))
    res = Parallel(n_jobs=n_jobs)(delayed(targets_chunk)(prices, c.tolist(), n_window, timeframe, cfg, max_assets_list, lag, dd_hl) for c in chunks)
    targets_list = [x for r in res for x in r]
    print(f"  pha 1: {len(ts)} mốc rebalance x {len(max_assets_list)} giá trị max_assets - {time.time() - t0:.0f}s")

    hold = "hold_" + cfg["hold_asset"]
    names = list(dict.fromkeys([n for tg in targets_list for n in tg] + ([hold] if cfg["hold_asset"] in pos else [])))
    row = {n: i for i, n in enumerate(names)}
    S, A = len(names), len(cols)
    Wm = np.zeros((S, A))                         # weight hiện tại (đã trôi theo giá) của từng strategy
    live = np.zeros(S, bool)                      # strategy đã có weight lần đầu chưa
    Rm = np.full((total, S), np.nan)
    rets = prices.pct_change().fillna(0).to_numpy()
    hist = {}
    targets_it = iter(targets_list)

    for k, t in enumerate(range(start, len(prices))):
        turn = np.zeros(S)
        if k % rebalance_every == 0:
            targets = next(targets_it)
            if k == 0 and cfg["hold_asset"] in pos:
                targets[hold] = {cfg["hold_asset"]: 1.0}   # buy & hold, không rebalance
            for name, tw in targets.items():
                i = row[name]
                vec = np.zeros(A)
                for a, x in tw.items():
                    vec[pos[a]] = x
                if live[i] and np.abs(vec - Wm[i]).max() < threshold:
                    continue                      # lệch chưa đủ -> giữ nguyên, không phí
                turn[i] = np.abs(vec - Wm[i]).sum()
                Wm[i] = vec
                live[i] = True
                hist.setdefault(name, {})[prices.index[t]] = pd.Series(tw)
        r = rets[t]
        pr = Wm @ r
        Rm[k] = np.where(live, pr - fee * turn, np.nan)
        Wm *= 1 + r
        Wm /= (1 + pr)[:, None]

    returns = pd.DataFrame(Rm, index=prices.index[start:], columns=names).dropna(axis=1, how="all")
    return returns, {n: pd.DataFrame(h).T.fillna(0) for n, h in hist.items()}


def summarize(returns, timeframe, weights_hist):
    P = periods_per_year(timeframe)
    rows = {}
    for name, r in returns.items():
        eq = (1 + r).cumprod()
        sd = r.std()
        wh = weights_hist.get(name)
        # turnover/năm = tổng |Δweight| giữa các lần rebalance (không tính trôi giá)
        tn = wh.diff().abs().sum().sum() if wh is not None and len(wh) > 1 else 0.0
        rows[name] = {
            "return_cagr": eq.iloc[-1] ** (P / len(r)) - 1,
            "volatility": sd * np.sqrt(P),
            "sharpe": (r.mean() - RF / P) / sd * np.sqrt(P) if sd > 0 else 0.0,
            "max_drawdown": (eq / eq.cummax() - 1).min(),
            "turnover_per_year": tn * P / len(r),
        }
    return pd.DataFrame(rows).T.sort_values("sharpe", ascending=False)


def filter_listed(symbols, quote, start_date, require_full_history=True, binance="https://api.binance.com"):
    """Giữ (theo thứ tự đầu vào) các symbol có cặp spot {symbol}{quote} trên Binance và, nếu
    require_full_history, có nến đầu tiên <= start_date (đủ lịch sử cho cả cửa sổ fit + backtest).
    Lọc TRƯỚC khi tải/rolling để mã niêm yết muộn (vd HYPE) không bị loại âm thầm ở từng cửa sổ."""
    def first_candle(sym):
        try:
            r = requests.get(f"{binance}/api/v3/klines", timeout=10,
                             params=dict(symbol=sym + quote, interval="1d", startTime=0, limit=1))
            data = r.json() if r.status_code == 200 else []
            return pd.to_datetime(data[0][0], unit="ms").date() if data else None
        except Exception:
            return None

    with ThreadPoolExecutor(max_workers=8) as ex:
        firsts = list(ex.map(first_candle, symbols))
    kept = []
    for s, f in zip(symbols, firsts):
        if f is None:
            print(f"  ✗ {s}: không có cặp {s}{quote} trên Binance spot")
        elif require_full_history and f > start_date:
            print(f"  ✗ {s}: niêm yết {f} > {start_date} (thiếu lịch sử)")
        else:
            kept.append(s)
    return kept


In [ ]:
# ====================== PARAMS 1/2: DỮ LIỆU & BACKTEST ======================

# --- Khung thời gian & độ dài backtest ---
timeframe = '1h'            # Khung nến Binance: 1m 5m 15m 30m 1h 2h 4h 6h 8h 12h 1d 1w
display_candles = 10000     # Số nến backtest được hiển thị/đánh giá (phần out-of-sample)
n_window = display_candles  # Số nến lịch sử liền trước dùng để fit weight ở mỗi bước.
                            # Tổng nến tải về = display_candles + n_window (mặc định gấp đôi)

# --- Tần suất & chi phí ---
n_jobs = -1                 # Số tiến trình chạy song song tính target (mỗi mốc rebalance chạy độc lập, gồm cả grid max_assets).
                            # -1 = dùng mọi core, 1 = tuần tự, 4 = tối đa 4 core
rebalance_days = 1          # Chu kỳ tính lại weight, tính bằng NGÀY (30 = hàng tháng, 7 = hàng tuần, 1 = hàng ngày).
                            # Tự quy ra số candle theo timeframe (1h -> 720, 1d -> 30). Giữa 2 lần
                            # rebalance weight trôi theo giá. Nhỏ hơn 1 candle -> mỗi candle (chậm nhất)
rebalance_threshold = 0.01  # Chỉ rebalance khi weight lệch >= ngưỡng này (điểm % tuyệt đối của bất kỳ asset nào,
                            # 0.01 = 1%) so với target mới, kiểm tra tại mỗi mốc rebalance_days.
                            # 0 = luôn rebalance; chưa đủ lệch -> giữ nguyên, không tốn phí
data_lag_days = 1           # HARD-RULE độ trễ dữ liệu (ngày): hôm nay là T chỉ có data đến T-1, nên weight tối ưu
                            # từ data đó chỉ ảnh hưởng từ T+1. Quy ra candle theo timeframe. 0 = tắt (chỉ để so sánh)
fee = 0.001                 # Phí mỗi lần giao dịch, tính trên turnover (0.001 = 0.1%)
RF = 0.03                   # Lãi suất phi rủi ro / năm, dùng cho max-Sharpe VÀ cho chỉ số Sharpe

# --- Vũ trụ tài sản ---
top_n_coins = 8             # Lấy top N coin theo vốn hóa từ CoinGecko (đã lọc stablecoin, token wrapped)
candidate_pool = 30         # Lấy top N này theo vốn hóa rồi LỌC (bỏ mã không có cặp spot trên Binance hoặc thiếu lịch sử)
                            # cho đến khi đủ top_n_coins. Phải >= top_n_coins; tăng nếu nhiều mã bị loại
require_full_history = True # True = chỉ giữ mã đã niêm yết trước ngày bắt đầu dữ liệu (đủ cho cửa sổ fit + trễ + backtest)
quote_asset = 'USDT'        # Đồng định giá; ticker = symbol + quote_asset (vd BTC -> BTCUSDT)
extra_tickers = ['EURUSDT', 'PAXGUSDT']   # Tài sản thêm tay, luôn nằm trong vũ trụ (EUR, vàng)

# --- Tải dữ liệu ---
data_dir = 'binance_data'   # Thư mục lưu CSV Binance (gitignore)
download_workers = 15       # Số luồng tải song song
download_retries = 3        # Số lần thử lại mỗi file

In [ ]:
# ====================== PARAMS 2/2: CHIẾN LƯỢC ======================

# Grid test: chạy walk-forward riêng cho từng giá trị max_assets rồi so sánh.
# Chậm theo số giá trị (range(2, 11) = 9 lần). Thu hẹp (vd range(4, 7)) khi thử nhanh.
max_assets_range = range(2, 11)

cfg = dict(
    # --- Ràng buộc trọng số ---
    required_assets={'EURUSDT': 0.15, 'PAXGUSDT': 0.09, 'BTCUSDT': 0.06, 'ETHUSDT': 0.04},
                            # Mã bắt buộc có mặt + trọng số TỐI THIỂU. Tổng sàn phải <= 100%
    min_weight=0.01,        # Trọng số tối thiểu của các mã còn lại được chọn
    other_max_weight=0.10,  # Trọng số TỐI ĐA của mã không nằm trong required_assets (áp cho các strategy opt_*).
                            # Nếu tổng trần không đủ 100% (quá ít mã bắt buộc) thì bước tối ưu đó bị bỏ qua
    eur_assets={'EURUSDT', 'EURIUSDT'},   # Nhóm tài sản bị chặn trần bởi eur_max_weight
    eur_max_weight=0.34,    # Trọng số tối đa cho mỗi mã trong eur_assets

    # --- Mục tiêu tối ưu ---
    target_return=0.05,     # Return/năm mục tiêu cho strategy 'opt_optimal' (efficient_return).
                            # Không đạt được -> tự chuyển về min-volatility

    # --- Ước lượng đầu vào cho optimizer ---
    risk_model='exp_shrink',  # Ma trận hiệp phương sai: 'shrinkage' (Ledoit-Wolf) | 'sample' |
                              # 'exp' (EWMA) | 'exp_shrink' (50% EWMA + 50% Ledoit-Wolf)
    cov_span_days=60,       # Span EWMA tính theo ngày (chỉ dùng cho 'exp'/'exp_shrink')
    mu_shrink=0.5,          # Co return kỳ vọng về trung bình chéo: 0 = giữ nguyên mu thô,
                            # 1 = mọi mã cùng mu. Tăng để bớt nhiễu / bớt đuổi theo mã thắng

    # --- Làm tròn weight ---
    weight_step=0.01,       # Mọi target weight là bội số của bước này (0.01 = 1%, không số lẻ hơn). 0 = tắt

    # --- Benchmark ---
    mdd_targets=(-0.08, -0.13, -0.21, -0.34, -0.55),
                            # MDD mục tiêu cho các danh mục absolute_equal_dd_xx
    mdd_years=5,            # absolute_equal_dd lấy MDD từng mã trong N năm gần nhất (rolling, chỉ data đã biết)
                            # thay vì chỉ trong cửa sổ fit; mã niêm yết ngắn hơn N năm dùng toàn bộ lịch sử có
    mdd_timeframe='1d',     # Khung nến dùng tính MDD dài hạn (tải riêng, nhẹ hơn nhiều so với nến 1h).
                            # MDD = (Low - cummax(High)) / cummax(High): bắt cả đợt sụt trong ngày
    cash_asset='EURUSDT',   # Mã nhận phần dư của absolute_equal_dd (không tham gia ngân sách rủi ro)
    hold_asset='BTCUSDT',   # Mã dùng làm benchmark buy & hold
)

# --- Dẫn xuất (không cần sửa) ---
rebalance_every = max(1, round(rebalance_days * 86400 / tf_delta(timeframe).total_seconds()))   # số candle
lag_candles = round(data_lag_days * 86400 / tf_delta(timeframe).total_seconds())              # độ trễ, số candle
cfg['span'] = int(cfg['cov_span_days'] * 24 * 3600 / tf_delta(timeframe).total_seconds())

In [ ]:
# --- Tải & nạp dữ liệu ---
t_load = time.time()
end_date = date.today() - timedelta(days=1)
start_date, days = get_start_date(timeframe, display_candles + n_window + lag_candles + 24, end_date)   # +24 nến đệm
start_str, end_str = start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")
print(f"{timeframe} -> {days} days: {start_str} .. {end_str}")

# Lọc vũ trụ TRƯỚC khi tải/rolling: top theo vốn hóa nhưng phải có cặp spot và đủ lịch sử
print(f"Lọc top {candidate_pool} ứng viên:")
top_coins = filter_listed(get_top_coins(candidate_pool), quote_asset, start_date, require_full_history)[:top_n_coins]
if len(top_coins) < top_n_coins:
    print(f"⚠️ Chỉ có {len(top_coins)}/{top_n_coins} mã đạt điều kiện, tăng candidate_pool hoặc giảm top_n_coins")
tickers_binance = extra_tickers + [s + quote_asset for s in top_coins]
print("Tickers:", tickers_binance)

download_binance_klines_fixed_symbols(
    symbols=tickers_binance, intervals=[timeframe], time_period="daily",
    start=start_str, end=end_str, destination_dir=data_dir,
    max_workers=download_workers, retries=download_retries,
)

price_df = load_prices(tickers_binance, start_str, end_str, timeframe, data_dir)

# MDD dài hạn cho absolute_equal_dd: nến mdd_timeframe, cfg['mdd_years'] năm gần nhất
mdd_start = end_date - relativedelta(years=cfg['mdd_years'])
this_month = end_date.replace(day=1)
for period, s, e in (("monthly", mdd_start.strftime("%Y-%m"), (this_month - timedelta(days=1)).strftime("%Y-%m")),
                     ("daily", this_month.strftime("%Y-%m-%d"), end_str)):
    download_binance_klines_fixed_symbols(
        symbols=tickers_binance, intervals=[cfg['mdd_timeframe']], time_period=period, start=s, end=e,
        destination_dir=data_dir, max_workers=download_workers, retries=download_retries,
    )
dd_hl = tuple(load_prices(tickers_binance, mdd_start.strftime("%Y-%m-%d"), end_str, cfg['mdd_timeframe'], data_dir, col=k)
              for k in ("High", "Low"))               # đỉnh theo High, đáy theo Low -> bắt cả sụt trong ngày
print(f"MDD lookback: {len(dd_hl[0])} nến {cfg['mdd_timeframe']} ({cfg['mdd_years']} năm)")
price_df = price_df.iloc[-(display_candles + n_window + lag_candles):]   # cắt đúng số nến cần
n_window = min(n_window, len(price_df) - 1 - lag_candles)
print(f"{len(price_df)} candles = cửa sổ fit {n_window} + trễ {lag_candles} + backtest {len(price_df) - n_window - lag_candles}")
print("Số giá hợp lệ / mã:"); print(price_df.notna().sum().to_string())
print(f"⏱ Tải & nạp dữ liệu: {time.time() - t_load:.1f}s")

In [ ]:
# --- Chạy walk-forward (pha 1 song song theo mốc rebalance x grid max_assets, pha 2 vector hóa) + bảng kết quả ---
t_run = time.time()
returns_wf, weights_hist = walk_forward(price_df, n_window, timeframe, cfg, list(max_assets_range), rebalance_every=rebalance_every,
                                        fee=fee, threshold=rebalance_threshold, lag=lag_candles, dd_hl=dd_hl, n_jobs=n_jobs)
summary = summarize(returns_wf, timeframe, weights_hist)
print(f"⏱ Walk-forward (grid {len(max_assets_range)} giá trị max_assets, n_jobs={n_jobs}): {time.time() - t_run:.1f}s")
(summary * [100, 100, 1, 100, 1]).round(2)   # cột: return_cagr%, volatility%, sharpe, max_drawdown%, turnover/năm

In [ ]:
# --- Plot: equity của mọi strategy (net of fees) ---
fig = go.Figure()
for name, r in returns_wf.items():
    fig.add_trace(go.Scatter(x=r.index, y=(1 + r).cumprod(), mode="lines", name=name,
                             line=dict(width=3 if name.startswith(("opt_", "hold_")) else 1)))
fig.update_layout(title="Walk-forward equity (net of fees)", hovermode="x unified")
fig.show()

In [ ]:
# --- Strategy tốt nhất theo 1 chỉ số: weight hiện tại + equity + weight theo thời gian ---
def show_best(metric):
    best = summary[metric].idxmax()
    print(f"Best {metric}: {best} ({summary.loc[best, metric]:.4g})")
    wh = weights_hist[best]
    cur = wh.iloc[-1]
    print("Weight hiện tại (lần rebalance gần nhất):")
    print((cur[cur > 0].sort_values(ascending=False) * 100).round(2).astype(str) + "%")
    eq = (1 + returns_wf[best]).cumprod()
    go.Figure(go.Scatter(x=eq.index, y=eq, name=best)).update_layout(title=f"Equity - {best}").show()
    go.Figure([go.Scatter(x=wh.index, y=wh[c], stackgroup="w", name=c) for c in wh.columns]
              ).update_layout(title=f"Weights - {best}", hovermode="x unified").show()

show_best("return_cagr")

In [ ]:
show_best("sharpe")

In [ ]:
end_time = time.time()
elapsed_seconds = end_time - start_time

# Chuyển sang định dạng hh:mm:ss
hours, rem = divmod(elapsed_seconds, 3600)
minutes, seconds = divmod(rem, 60)

print(f"Thời gian xử lý: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")